In [0]:
df= spark.read.format("csv").option("header", True).option("inferSchema", True).load("/Volumes/ct_salesanalytics/schema_salesanalytics/vol_sales/sales.csv")

In [0]:
# Keep the loaded source unchanged until cleaning is complete.
# The validated Delta table is written in Cell 12.
print("Source loaded; continue with type conversion and validation.")

In [0]:
%python
# Inspect the loaded source schema without requiring a pre-existing table.
df.printSchema()

In [0]:
from pyspark.sql.functions import to_date, col

df = df.withColumn("sale_date", to_date(col("sale_date"), "M/d/yyyy"))
display(df)


In [0]:
from pyspark.sql.functions import col

df  = df.withColumn("quantity", col("quantity").cast("int"))\
    .withColumn("unit_price", col("unit_price").cast("double"))

In [0]:
df = df.dropDuplicates(["sale_id"])

In [0]:
df = df.withColumn("total_amount", col("quantity") * col("unit_price"))

In [0]:
# A null quantity is rejected rather than disappearing from both groups.
from pyspark.sql.functions import coalesce, lit
valid_quantity = coalesce(col("quantity") > 0, lit(False))
df_valid = df.filter(valid_quantity)
df_invalid = df.filter(~valid_quantity)

In [0]:
print("Total Records", df.count())
print("Valid Records", df_valid.count())
print("Invalid Records", df_invalid.count())

In [0]:
# CSV writes produce a directory of Spark part files, not one named CSV.
volume_root = "/Volumes/ct_salesanalytics/schema_salesanalytics/vol_sales"
df_valid.write.mode("overwrite").option("header", True).csv(
    f"{volume_root}/processed/valid/valid_sales"
)

In [0]:
df_invalid.write.mode("overwrite").option("header", True).csv(
    f"{volume_root}/processed/invalid/invalid_sales"
)

In [0]:
# Explicit batch replacement: this overwrites the project orders table when run.
df_valid.write.mode("overwrite").format("delta").option(
    "overwriteSchema", "true"
).saveAsTable("ct_salesanalytics.schema_salesanalytics.orders")

In [0]:
# Confirm the schema of the validated table after the write succeeds.
spark.table("ct_salesanalytics.schema_salesanalytics.orders").printSchema()